In [1]:
%load_ext autoreload
%autoreload 2

import IPython
from pathlib import Path
import os
locals = IPython.extract_module_locals() # type: ignore
notebook_name = "/".join(locals[1]["__vsc_ipynb_file__"].split("/"))
os.chdir(Path(notebook_name).parent.parent)

In [4]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from recsys_lakehouse.lakehouse.operator import TableOperator
from recsys_lakehouse.lakehouse.layers import Gold

In [5]:
spark = SparkSession.builder.appName("Raw").master("local[*]").getOrCreate()


In [14]:
gold_layer = Gold(dataset_name="amazon_books_from_2023-06")
operator = TableOperator(spark)
reviews_table = operator.read_table(gold_layer, gold_layer.tables["reviews"])
item_features_table = operator.read_table(gold_layer, gold_layer.tables["item_features"])
user_features_table = operator.read_table(gold_layer, gold_layer.tables["user_features"])


In [16]:
item_features_table.columns

['parent_asin', 'average_rating', 'price', 'main_category', 'categories']

In [17]:
user_features_table.columns

['user_id',
 'n_reviews',
 'mean_rating',
 'mean_helpful_vote',
 'std_helpful_vote']

In [13]:
reviews_table.columns

['timestamp', 'user_id', 'parent_asin', 'rating']

In [29]:
(
    reviews_table
    .withColumn("year_month_day", F.date_format("timestamp", "yyyy-MM-dd"))
    .groupBy('year_month_day')
    .count()
    .sort('year_month_day')
).show()

+--------------+-----+
|year_month_day|count|
+--------------+-----+
|    2023-07-01|  742|
|    2023-07-02|  845|
|    2023-07-03| 1010|
|    2023-07-04|  913|
|    2023-07-05|  855|
|    2023-07-06|  950|
|    2023-07-07|  915|
|    2023-07-08|  924|
|    2023-07-09|  884|
|    2023-07-10|  876|
|    2023-07-11|  972|
|    2023-07-12|  895|
|    2023-07-13|  860|
|    2023-07-14|  770|
|    2023-07-15|  789|
|    2023-07-16|  717|
|    2023-07-17|  916|
|    2023-07-18|  886|
|    2023-07-19|  854|
|    2023-07-20|  806|
+--------------+-----+
only showing top 20 rows



In [12]:
# Add a new column 'year_week' to the dataframe
reviews_table_with_year_week = (
    reviews_table
    .withColumn('year_week', F.concat(F.year('timestamp'), F.lit('-'), F.weekofyear('timestamp')))
    .groupBy('year_week')
    .count()
    .sort('year_week')
)
reviews_table_with_year_week.show()

+---------+-----+
|year_week|count|
+---------+-----+
|  2023-26| 1587|
|  2023-27| 6451|
|  2023-28| 5879|
|  2023-29| 5469|
|  2023-30| 4538|
|  2023-31| 4405|
|  2023-32| 4675|
|  2023-33| 4248|
|  2023-34| 4206|
|  2023-35| 2756|
|  2023-36|  794|
|  2023-37|   44|
+---------+-----+



In [23]:
item_features_table.groupBy("main_category").agg(F.mean("price").alias("mean_price")).dropna().sort("mean_price").show()

+-------------------+------------------+
|      main_category|        mean_price|
+-------------------+------------------+
|       Buy a Kindle|3.3043670053061773|
|       Toys & Games|12.643333117167154|
|    Office Products|17.989999771118164|
|        Amazon Home| 18.08999991416931|
|              Books| 18.45987659779195|
|Musical Instruments|25.190000534057617|
|     AMAZON FASHION|              45.0|
+-------------------+------------------+



In [26]:
item_features_table.groupBy("main_category").agg(F.mean("average_rating").alias("mean_rating")).dropna().sort("mean_rating").show()

+-------------------+------------------+
|      main_category|       mean_rating|
+-------------------+------------------+
|       Buy a Kindle| 4.473101586616405|
| Audible Audiobooks| 4.474768513606654|
|              Books| 4.577818792351475|
|    Office Products| 4.599999904632568|
|        Amazon Home|4.6499998569488525|
|       Toys & Games| 4.666666666666667|
|Musical Instruments| 4.800000190734863|
|     AMAZON FASHION|               5.0|
+-------------------+------------------+

